# XGBoost de Lucie (par arc)

Repris de `modele_xgboost_par_axe.ipynb` (branche `lucie/xgboost`) sans modification : un modèle par arc, `axe` retiré des features, `vacances` en one-hot, mêmes hyperparamètres.

In [ ]:
import sys; sys.path.append("..")  # pour importer dataset_create depuis notebooks/
import numpy as np
import pandas as pd
from dataset_create import charger, split, ORIGINES, FEATURES

df = charger(imputer=True)  # imputer=False : trous laissés en NaN
rmse = lambda y, p, axe: ((y - p) ** 2).groupby(axe, observed=True).mean() ** .5  # RMSE par arc

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

model_features = [f for f in FEATURES if f != "axe"]
categorical_features = ["vacances"]
numeric_features = [f for f in model_features if f not in categorical_features]

def create_model():  # copie de son notebook
    preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
                                                   ("num", "passthrough", numeric_features)])
    regressor = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             objective="reg:squarederror", random_state=42, n_jobs=-1)
    return Pipeline([("preprocessor", preprocessor), ("xgb", regressor)])

def modele(X_train, y_train, X_test):
    p = pd.Series(np.nan, index=X_test.index)
    for axe in X_test.axe.unique():  # un modèle par arc
        tr, te = X_train.axe == axe, X_test.axe == axe
        p[te] = create_model().fit(X_train.loc[tr, model_features], y_train[tr]).predict(X_test.loc[te, model_features])
    return p.values

In [ ]:
# 1 semaine : dernière donnée le ven. 07/11/2025 -> on prédit sam. 08 -> mer. 12/11 (11/11 férié inclus)
CIBLE = "debit"  # ou "occupation"
X_train, X_test, y_train, y_test = split(df, "2025-11-07", cible=CIBLE)
p = modele(X_train, y_train, X_test)
print(pd.DataFrame({"modele": rmse(y_test, p, X_test.axe), "naif_lag7j": rmse(y_test, X_test.lag_7j, X_test.axe)}).round(1))

In [ ]:
# Les 3 tests communs, contre le naïf lag_7j : semaine de Lucie (07/11/2025), test final, backtest sur TOUS les vendredis
from evaluation import trois_tests
P = trois_tests(df, modele, "xgboost", CIBLE)